# Pipeline de Auditoria de Credenciamento
Este notebook orquestra as 5 fases do pipeline analítico. Ele foi adaptado para permitir a execução passo a passo, facilitando a inspeção dos dados (DataFrames) em cada etapa do funil antes de gerar o relatório executivo.

### Configuração de Ambiente e Importações
Inicializa os módulos de log, carrega as chaves de API (.env) e importa os scripts de cada fase.

In [1]:
import logging
import inspect
import pandas as pd
from dotenv import load_dotenv
import nest_asyncio

# Permite rodar rotinas assíncronas (como o Gemini) dentro do Jupyter Notebook
nest_asyncio.apply()

# Carrega variáveis de ambiente (GEMINI_API_KEY)
load_dotenv()

# Configuração global de log do orquestrador
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

# Importações dos módulos locais do pipeline
from fase_1 import executar_fase_1
from fase_2 import executar_fase_2
from fase_3 import executar_fase_3
from fase_4 import executar_fase_4
from fase_5 import executar_fase_5

# Arquivo que será ingerido pelo pipeline
ARQUIVO_FONTE = '../Base dedados de Licitação em 17-09.xlsx' # Mude para o caminho correto caso necessário

c:\Users\akendall\Downloads\analise-licitacoes\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


### FASE 1: Ingestão e Higienização
Carrega os dados brutos, valida a chave primária e padroniza as colunas de texto e valores financeiros.

In [6]:
logging.info('Iniciando Fase 1: Ingestão e Higienização...')

df_analytical, df_raw = executar_fase_1(ARQUIVO_FONTE)

print(f'\nTotal de registros únicos: {len(df_analytical)}')
df_analytical.head()

[INFO] Iniciando Fase 1: Ingestão e Higienização...
[INFO] Registros iniciais: 46476 | Protocolos únicos: 23386
[INFO] Fase 1 concluída com sucesso. Base analítica consolidada e higienizada.



Total de registros únicos: 23386


,Protocolo,Categoria,Subcategoria,Setor Atual,No.Licitação,Julgado?,Município,Jurisdicionado,Modalidade,Tipo do Objeto,...,Valor,Valor Contrato,Valor Aditivo,GI - Resultado Etapa 1,GI - Grau de Evidência,GI - Fundamentação Contextual,GI - Divergência Cadastral Aparente,GI - Corroboração Utilizada,GI - Protocolo Relacionado,GI - Regime Jurídico Informado
0,Doc. 00011/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00001/2025,Não,PEDRA LAVRADA,CAMARA MUNICIPAL DE PEDRA LAVRADA,DISPENSA (LEI No 14.133/2021),Obras e Serviços de engenharia,...,105047.07,105047.07,0.0,NÃO CLASSIFICADO,<NA>,<NA>,False,<NA>,<NA>,Lei 14.133/2021
1,Doc. 00016/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00019/2025,Não,CACIMBA DE AREIA,PREFEITURA MUNICIPAL DE CACIMBA DE AREIA,INEXIGIBILIDADE (LEI No 14.133/2021),Compras e Serviços,...,100000.00,100000.00,0.0,NÃO CLASSIFICADO,<NA>,<NA>,False,<NA>,<NA>,Lei 14.133/2021
2,Doc. 00018/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00014/2025,Não,LAGOA,CAMARA MUNICIPAL DE LAGOA,DISPENSA (LEI No 14.133/2021),Compras e Serviços,...,21250.00,21250.00,0.0,NÃO CLASSIFICADO,<NA>,<NA>,False,<NA>,<NA>,Lei 14.133/2021
3,Doc. 00020/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00014/2025,Não,JACARAU,CAMARA MUNICIPAL DE JACARAU,DISPENSA (LEI No 14.133/2021),Compras e Serviços,...,48000.00,48000.00,0.0,NÃO CLASSIFICADO,<NA>,<NA>,False,<NA>,<NA>,Lei 14.133/2021
4,Doc. 00022/26,Licitações e Contratos,Licitações,CARTÓRIO DIAFI,00009/2025,Não,CONDADO,PREFEITURA MUNICIPAL DE CONDADO,ADESAO A ATA DE REGISTRO DE PRECOS (LEI No 14....,Obras e Serviços de engenharia,...,785712.32,785712.32,0.0,NÃO CLASSIFICADO,<NA>,<NA>,False,<NA>,<NA>,Lei 14.133/2021


In [3]:
df_raw.columns

Index(['Protocolo', 'Categoria', 'Subcategoria', 'Setor Atual', 'No.Licitação',
       'Julgado?', 'Município', 'Jurisdicionado', 'Modalidade',
       'Tipo do Objeto', 'Objeto', 'Estágio', 'Fracassada?',
       'CPF/CNPJ Proponente', 'Proponente', 'Situação Proposta',
       'Ano Homologação', 'Valor Estimado', 'Valor Proposta',
       'Valor Licitação', 'Valor Contrato', 'Valor Aditivo'],
      dtype='str')

### FASE 2: Triagem Determinística e Vetorial
Filtra os 'lixos óbvios', identifica menções incidentais e isola a 'zona cinzenta' (busca léxica e similaridade semântica local) para a fila da IA.

In [7]:
logging.info('Iniciando Fase 2: Triagem Determinística e Peneira Vetorial...')

df_analytical, df_queue = executar_fase_2(df_analytical)

print(f'\nTamanho da fila separada para avaliação do LLM: {len(df_queue)}')
df_queue.head()

[INFO] Iniciando Fase 2: Triagem Determinística e Peneira Vetorial...
[INFO] Processando 15184 registros na Peneira Fina Semântica...
[INFO] Use pytorch device_name: cpu
[INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
c:\Users\akendall\Downloads\analise-licitacoes\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Batches: 100%|██████████| 475/475 [07:49<00:00,  1.01it/s]
[INFO] Fase 2 concluída. Descartados: 7685 | Direcionados para LLM: 15701



Tamanho da fila separada para avaliação do LLM: 15701


,Protocolo,Modalidade,Objeto,GI - Regime Jurídico Informado,Valor
1,Doc. 00016/26,INEXIGIBILIDADE (LEI No 14.133/2021),APRESENTACAO DE UM SHOW ARTISTICO DA DUPLA EDY...,Lei 14.133/2021,100000.0
2,Doc. 00018/26,DISPENSA (LEI No 14.133/2021),"CONTRATACAO DE EMPRESA PARA PLANEJAMENTO, ELAB...",Lei 14.133/2021,21250.0
3,Doc. 00020/26,DISPENSA (LEI No 14.133/2021),"CONTRATACAO DE SERVICOS DE DIGITALIZACAO, CATA...",Lei 14.133/2021,48000.0
8,Doc. 00030/26,DISPENSA (LEI No 14.133/2021),CONTRATACAO DE SERVICOS DE PROTOCOLO ELETRONIC...,Lei 14.133/2021,30000.0
9,Doc. 00033/26,PREGAO (LEI No 14.133/2021),CONTRATACAO DE EMPRESA ESPECIALIZADA PARA EXEC...,Lei 14.133/2021,471670.0


In [8]:
df_analytical.to_csv('df_analytical_fase2.csv', index=False)
df_queue.to_csv('df_queue_fase2.csv', index=False)

### FASE 3: Análise Semântica (LLM - Gemini)
Envia a fila isolada na fase anterior para avaliação de mérito utilizando o modelo Gemini (assíncrono), extraindo o raciocínio e possíveis divergências cadastrais da Prefeitura.

In [3]:
df_analytical_sample = pd.read_excel('df_analytical_sample.xlsx')
df_queue_sample = pd.read_excel('df_queue_sample.xlsx')

Usando só uma sample do df completo:

In [4]:
"""
df_queue_sample = df_queue.sample(n=100, random_state=42)
df_analytical_sample = df_analytical[df_analytical['Protocolo'].isin(df_queue_sample['Protocolo'])].copy()

df_queue_sample.to_excel('df_queue_sample.xlsx', index=False)
df_analytical_sample.to_excel('df_analytical_sample.xlsx', index=False)
"""

"\ndf_queue_sample = df_queue.sample(n=100, random_state=42)\ndf_analytical_sample = df_analytical[df_analytical['Protocolo'].isin(df_queue_sample['Protocolo'])].copy()\n\ndf_queue_sample.to_excel('df_queue_sample.xlsx', index=False)\ndf_analytical_sample.to_excel('df_analytical_sample.xlsx', index=False)\n"

In [ ]:
logging.info('Iniciando Fase 3: Julgamento Semântico via LLM (Assíncrono)...')

# Como o Jupyter tem seu próprio event loop, podemos usar wait diretamente na célula
if inspect.iscoroutinefunction(executar_fase_3):
    df_analytical_fase3 = await executar_fase_3(df_analytical_sample, df_queue_sample)
else:
    df_analytical_fase3 = executar_fase_3(df_analytical_sample, df_queue_sample)

# Mostra o balanço de como os dados foram classificados
display(df_analytical_fase3[['GI - Resultado Etapa 1', 'GI - Grau de Evidência']].value_counts())

[INFO] Iniciando Fase 3: Julgamento Semântico via LLM (Assíncrono)...
[INFO] [CHECKPOINT] Nenhum histórico prévio encontrado. Total na fila: 100 registros.
[INFO] [MODO PAGO ATIVO] Modelo: gemini-2.5-flash | Concorrência paralela: 10 tarefas.
[INFO] Iniciando inferência de 100 registros pendentes no Gemini API (gemini-2.5-flash)...
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] AFC is enabled with max remote calls: 10.
[INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 404 Not Found"
[INFO] HTTP Request: POST https://ge

In [5]:
df_analytical_fase3.to_csv('df_analytical_fase3.csv', index=False)

In [8]:
df_analytical_fase3.to_excel('df_analytical_fase3.xlsx', index=False)

In [10]:
quebra

NameError: name 'quebra' is not defined

### FASE 4: Corroboração Interna e Testes de Integridade
Aplica heurísticas para identificar clusters de mercado fluido (Promoção para Grau A) e executa os testes vitais de consistência dos dados.

In [ ]:
logging.info('Iniciando Fase 4: Corroboração Interna e Asserções de Qualidade...')

df_analytical = executar_fase_4(df_analytical)

# Mostra as contratações irmãs promovidas (se houver)
mask_promovidos = df_analytical['GI - Corroboração Utilizada'].notna()
display(df_analytical[mask_promovidos][['Protocolo', 'Município', 'Modalidade', 'GI - Corroboração Utilizada', 'GI - Grau de Evidência']])

### FASE 5: Geração do Entregável Executivo
Salva o arquivo final (.xlsx) contendo as formatações condicionais de alerta, as quebras de linha e o roteamento de abas.

In [ ]:
logging.info('Iniciando Fase 5: Geração do Relatório Executivo Excel...')

nome_arquivo_excel = executar_fase_5(df_analytical, df_raw)

print(f'\n✅ Pipeline finalizado! Relatório pronto em: {nome_arquivo_excel}')